# MedTube Segmentation — YOLO11n-RGBD (4-channel) on Colab GPU

Trains YOLO11n-seg on 4-channel RGB-D images (RGB + depth as 4th channel).
Dataset: `rgbd_split/` — 2100 train / 450 valid / 450 test (70/15/15, seed=42).

**Before running:**
1. Runtime → Change runtime type → **GPU (T4 or A100)**
2. Locally run `tools/split_rgbd.py` to create `rgbd_split/`
3. Zip `rgbd_split/` → `rgbd_split.zip` and upload to `MyDrive/2026/`
4. Upload `weights/yolo11n.pt` to `MyDrive/2026/`
5. Run all cells top to bottom

**Timeout protection:**
- Cell 2: JavaScript anti-idle (clicks every 60 s — keeps browser session alive)
- Cell 5: Python keep-alive thread (belt-and-suspenders)
- `save_period=10` saves a checkpoint to Drive every 10 epochs
- Cell 11: Resume from last checkpoint if session disconnects

In [ ]:
# Cell 1 — Install
%pip install ultralytics -q

In [ ]:
# Cell 2 — Anti-timeout: JavaScript click simulation
# Simulates a periodic click on the Colab "Connect" button so the
# browser session never registers as idle. Runs silently in the background.
from IPython.display import Javascript, display

display(Javascript("""
function clickConnect() {
  const btn = document.querySelector("colab-connect-button");
  if (btn) btn.click();
}
setInterval(clickConnect, 60000);  // every 60 seconds
console.log("Anti-timeout JS active");
"""))
print("Anti-timeout JS injected — clicking Connect button every 60 s.")

In [ ]:
# Cell 2 — Check GPU
!nvidia-smi
import torch
print(f"CUDA: {torch.cuda.is_available()}, Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 3 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 4 — Unzip pre-split dataset from Google Drive
# Upload rgbd_split.zip (created locally by tools/split_rgbd.py) to MyDrive/2026/
import os
import shutil
import zipfile

ZIP_PATH   = '/content/drive/MyDrive/2026/rgbd_split.zip'
RGBD_SPLIT = '/content/rgbd_split'

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        f"Zip not found at {ZIP_PATH}\n"
        "Upload rgbd_split.zip to Google Drive → MyDrive/2026/ first."
    )

print("Extracting dataset…")
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('/content/')

# Fix nested structure: zip -r sometimes produces rgbd_split/rgbd_split/
if not os.path.exists(f'{RGBD_SPLIT}/train') and \
        os.path.exists(f'{RGBD_SPLIT}/rgbd_split/train'):
    print("Detected nested directory — flattening…")
    nested = f'{RGBD_SPLIT}/rgbd_split'
    for item in os.listdir(nested):
        shutil.move(f'{nested}/{item}', f'{RGBD_SPLIT}/{item}')
    os.rmdir(nested)

for split in ('train', 'valid', 'test'):
    path = f'{RGBD_SPLIT}/{split}/images'
    if os.path.exists(path):
        print(f"  {split:6s}: {len(os.listdir(path))} images")
    else:
        raise RuntimeError(f"Expected {path} — check zip structure.")

In [ ]:
# Cell 5 — Keep-alive (prevents Colab idle timeout)
import threading
import time

def keep_alive():
    while True:
        time.sleep(60)

thread = threading.Thread(target=keep_alive, daemon=True)
thread.start()
print("Keep-alive active — session will not idle-timeout")

In [ ]:
# Cell 6 — Write data.yaml for the pre-split RGBD dataset
import os

RGBD_SPLIT = '/content/rgbd_split'

# Ensure directories exist (safety net if Cell 4 had issues)
for split in ('train', 'valid', 'test'):
    os.makedirs(f'{RGBD_SPLIT}/{split}/images', exist_ok=True)
    os.makedirs(f'{RGBD_SPLIT}/{split}/labels', exist_ok=True)

data_yaml = f"""train: {RGBD_SPLIT}/train/images
val:   {RGBD_SPLIT}/valid/images
test:  {RGBD_SPLIT}/test/images

nc: 4
names: ['Other', 'Push-on', 'Screwcap', 'Universal']
"""

yaml_path = f'{RGBD_SPLIT}/data.yaml'
with open(yaml_path, 'w') as f:
    f.write(data_yaml)

print(f"data.yaml written → {yaml_path}")
for split in ('train', 'valid', 'test'):
    n = len(os.listdir(f'{RGBD_SPLIT}/{split}/images'))
    print(f"  {split:6s}: {n} images")

In [ ]:
# Cell 7 — Patch YOLO11n for 4-channel input (RGB + depth)
# Copies the pretrained yolo11n.pt from Drive, then patches the first conv
# to accept 4 channels (depth channel initialised to zero).
import copy
import shutil
import torch
from ultralytics import YOLO

DRIVE_WEIGHTS = '/content/drive/MyDrive/2026/yolo11n.pt'
LOCAL_WEIGHTS = '/content/yolo11n.pt'

shutil.copy2(DRIVE_WEIGHTS, LOCAL_WEIGHTS)
print(f"Copied weights from Drive.")

model = YOLO(LOCAL_WEIGHTS)

first_conv = model.model.model[0].conv  # type: ignore[union-attr]
old_weight  = first_conv.weight.data    # [out_ch, 3, kH, kW]

new_conv = torch.nn.Conv2d(
    4, old_weight.shape[0],
    kernel_size=first_conv.kernel_size,
    stride=first_conv.stride,
    padding=first_conv.padding,
    bias=first_conv.bias is not None,
)
with torch.no_grad():
    new_conv.weight[:, :3] = old_weight   # keep RGB pretrained weights
    new_conv.weight[:, 3:]  = 0.0         # depth channel: zero-init
    if first_conv.bias is not None:
        new_conv.bias = copy.deepcopy(first_conv.bias)

model.model.model[0].conv = new_conv          # type: ignore[union-attr]
model.model.model[0].conv.in_channels = 4    # type: ignore[union-attr]
if hasattr(model.model, 'yaml'):              # type: ignore[union-attr]
    model.model.yaml['ch'] = 4               # type: ignore[union-attr]

print(f"Conv patched: 3ch → 4ch | weight: {new_conv.weight.shape}")

In [ ]:
# Cell 8 — Train
RGBD_SPLIT  = '/content/rgbd_split'
DRIVE_SAVE  = '/content/drive/MyDrive/2026/medtube_runs'

results = model.train(
    data        = f'{RGBD_SPLIT}/data.yaml',
    epochs      = 100,
    imgsz       = 640,
    batch       = 16,
    patience    = 50,
    save_period = 10,       # checkpoint to Drive every 10 epochs
    workers     = 2,
    device      = 0,        # GPU
    project     = DRIVE_SAVE,
    name        = 'YOLO11n-RGBD',
    exist_ok    = True,
    # Disable colour augments — depth channel has no colour information
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
    degrees=20.0, translate=0.05, scale=0.25,
    shear=2.0, perspective=0.0001,
    flipud=0.0, fliplr=0.5,
    mosaic=0.2, erasing=0.2,
    auto_augment='randaugment',
)

print(f'\nDone. Best weights: {DRIVE_SAVE}/YOLO11n-RGBD/weights/best.pt')

In [ ]:
# Cell 9 — Evaluate on held-out test split
DRIVE_SAVE  = '/content/drive/MyDrive/2026/medtube_runs'
RGBD_SPLIT  = '/content/rgbd_split'

from ultralytics import YOLO

best = YOLO(f'{DRIVE_SAVE}/YOLO11n-RGBD/weights/best.pt')
r = best.val(data=f'{RGBD_SPLIT}/data.yaml', split='test', imgsz=640, device=0)
print(f'Box  mAP50={r.box.map50:.4f}  mAP50-95={r.box.map:.4f}')
print(f'Mask mAP50={r.seg.map50:.4f}  mAP50-95={r.seg.map:.4f}')

In [ ]:
# Cell 10 — Download
from google.colab import files
files.download(f'{DRIVE_SAVE}/YOLO11n-RGBD/weights/best.pt')

In [ ]:
# Cell 11 — Resume after disconnect
# Run Cells 1-7 first to reinstall deps, remount Drive, re-patch model,
# then run this cell instead of Cell 8 to continue from the last checkpoint.
import os
from ultralytics import YOLO

DRIVE_SAVE = '/content/drive/MyDrive/2026/medtube_runs'
LAST_PT    = f'{DRIVE_SAVE}/YOLO11n-RGBD/weights/last.pt'

if os.path.exists(LAST_PT):
    model = YOLO(LAST_PT)
    model.train(resume=True)
    print(f'Resumed. Best: {DRIVE_SAVE}/YOLO11n-RGBD/weights/best.pt')
else:
    print(f'No checkpoint at {LAST_PT} — run Cell 8 for a fresh start')